In [ ]:
from pathlib import Path
from torchvision import datasets, transforms

import torch
import time
import pybdm
import numpy as np

seed = 42

# For CUDA
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

data_dir = Path("data")

resize_transform = transforms.Compose([
    transforms.Resize((10, 10)),
    transforms.ToTensor()
])

train_data_resized = datasets.MNIST(
    root=data_dir,
    train=True,
    download=True,
    transform=resize_transform,
)


pixels = []
for image, _ in train_data_resized:
    pixels.append(image.view(-1))

pixels = torch.cat(pixels)
mean = pixels.mean().item()
std = pixels.std(correction=0).item()

print(f"Dataset pixel mean: {mean:.6f}")
print(f"Dataset pixel std:  {std:.6f}")

transform = transforms.Compose([
    transforms.Resize((10, 10)),
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))
])

train_data = datasets.MNIST(
    root=data_dir,
    train=True,
    download=True,
    transform=transform,
)

test_data = datasets.MNIST(
    root=data_dir,
    train=False,
    download=True,
    transform=transform,
)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit

## Splitting training and evaluation data
# Data is stratified in regards to label to ensure equal representation of labels as in Sakabe et al. (2026)
labels = train_data.targets.numpy()

pool_indices, val_indices = train_test_split(
    np.arange(len(train_data)),
    test_size = 10000,
    stratify=labels,
    random_state=seed
)

# Antagelse: samme for alle arkitekturer
splitter = StratifiedShuffleSplit(n_splits = 200, train_size=25000, random_state = seed)

training_splits = [
    pool_indices[train_index]
    for train_index, _ in splitter.split(pool_indices, labels[pool_indices])
]


In [ ]:
#Caching block, if needed copy this into the other versions
from torch.utils.data import TensorDataset, Subset

## Function for caching dataset in memory
def cache_dataset(dataset):
    # Transform once per image
    samples = [dataset[i] for i in range(len(dataset))]

    images = torch.stack([image for image, _ in samples])
    labels = torch.tensor(
        [label for _, label in samples],
        dtype=torch.long
    )

    return TensorDataset(images, labels)


In [ ]:
from torch import nn
from torch.nn import functional as F


def binarize(x):
   
    binary = torch.where(x > 0, 1.0, -1.0)
    clipped = x.clamp(-1, 1)
    return clipped + (binary - clipped).detach()


class BinaryLinear(nn.Linear):

    def forward(self, x):
        return F.linear(x, binarize(self.weight), self.bias)


class BinarizedMLP(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.fc1 = BinaryLinear(100, 8, bias=False)
        self.bn1 = nn.BatchNorm1d(8, affine=False)
        self.fc2 = BinaryLinear(8, 4, bias=False)
        self.bn2 = nn.BatchNorm1d(4, affine=False)
        self.fc3 = BinaryLinear(4, 10, bias=False)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = binarize(self.bn1(self.fc1(x)))
        x = binarize(self.bn2(self.fc2(x)))
        return self.fc3(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BinarizedMLP().to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
from torch.utils.data import DataLoader
from pybdm.partitions import PartitionIgnore
import pandas as pd

# Load data
train_data_cached = cache_dataset(train_data)
test_data_cached = cache_dataset(test_data)


# Experimental setup parameters
N_MODELS = 200

# Training hyperparameters (from Sakabe et al. (2026))
learning_rate = 0.001
batch_size = 128
patience = 5
max_epochs = 100


val_data = Subset(train_data_cached, val_indices.tolist())
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data_cached, batch_size=batch_size, shuffle=False)
criterion = nn.CrossEntropyLoss()
bdm_estimator = pybdm.BDM(ndim=2, shape=(4, 4), partition=PartitionIgnore) # Assumption: Partition ignore, ignores incomplete blocks

results_dir = Path("results") / "mnist_8_4"
results_dir.mkdir(parents=True, exist_ok=True)
run_summaries = []

# Measure BDM and entropy
def measure_complexity():
    bdm_value, entropy_value = 0.0, 0.0
    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            binary_weights = (layer.weight.detach().cpu().numpy() > 0).astype(int)
            counts = bdm_estimator.decompose_and_count(binary_weights) 
            bdm_value += bdm_estimator.compute_bdm(counts)
            entropy_value += bdm_estimator.compute_ent(counts)
    return float(bdm_value), float(entropy_value)


def evaluate_validation_loss():
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            total_loss += criterion(model(images), labels).item() * labels.size(0)
    model.train()
    return total_loss / len(val_loader.dataset)

# Initialize random seed for reproducibility
torch.manual_seed(seed)

for model_index in range(N_MODELS):

    train_indices = training_splits[model_index]
    train_subset = Subset(train_data_cached, train_indices)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    model = BinarizedMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    print(f"\nModel {model_index + 1}/{N_MODELS}")

    history = {
        "train_loss": [], "val_loss": [], "bdm": [], "entropy": [],
    }
    best_val_loss = float("inf")
    best_epoch_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        epoch_start = time.perf_counter()
        model.train()
        train_loss_perstep = []
        bdm_perstep = []
        entropy_perstep = []
        validation_loss_perstep = []
        step = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            if step % 20 == 0:
                val_loss = evaluate_validation_loss()
                validation_loss_perstep.append(val_loss)
            bdm_value, entropy_value = measure_complexity()
            loss.backward()
            optimizer.step()
            
            train_loss_perstep.append(loss.item())
            bdm_perstep.append(bdm_value)
            entropy_perstep.append(entropy_value)
            step += 1

        # Average each metric's measurements within this epoch
        history["train_loss"].append(float(np.mean(train_loss_perstep)))
        history["bdm"].append(float(np.mean(bdm_perstep)))
        history["entropy"].append(float(np.mean(entropy_perstep)))
        epoch_val_loss = float(np.mean(validation_loss_perstep))
        history["val_loss"].append(epoch_val_loss)
        print(
            f"Epoch {epoch:3d}: train = {history['train_loss'][-1]:.4f}, "
            f"val = {epoch_val_loss:.4f}, BDM = {history['bdm'][-1]:.2f}, "
            f"entropy = {history['entropy'][-1]:.4f}"
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        print(f"Full epoch time: {time.perf_counter() - epoch_start:.2f}s")

        if epoch_val_loss < best_epoch_val_loss:
            best_epoch_val_loss = epoch_val_loss
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping. Best checkpoint: epoch {best_epoch}")
                break


    model.load_state_dict(best_state)
    model.eval()
    
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
    test_accuracy = correct / len(test_data_cached)
    print(f"Test accuracy: {100 * test_accuracy:.2f}%")

    analysis_epochs = len(history["train_loss"]) - epochs_without_improvement
    results_path = results_dir / f"model_{model_index + 1:03d}.pt"
    torch.save({
        "history": history,
        "analysis_epochs": analysis_epochs,
        "test_accuracy": test_accuracy,
        "split_index": model_index,
        "best_epoch": best_epoch
    }, results_path)
    print(f"Saved epoch averages and test accuracy to {results_path}")

    epoch_results = pd.DataFrame(history)
    epoch_results.insert(0, "epoch", range(1, len(epoch_results) + 1))
    epoch_results.to_csv(results_dir / f"{model_index + 1:03d}.csv") 

    run_summaries.append({
        "split_index": model_index,
        "test_accuracy": test_accuracy,
        "results_path": str(results_path),
    })

torch.save(run_summaries, results_dir / "summary.pt")
print(f"Completed {N_MODELS} models. Results saved to {results_dir}")


In [ ]:
print(type(train_data_cached))
print(len(Subset(train_data_cached, training_splits[0])))